## Courier Charges Verification (ShopX) Project 

#### Import libraries for data handling and calculations.

In [1]:
import pandas as pd
import numpy as np
import math


C:\Users\upadh\anaconda3\anaconda2\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


#### Load all input files

In [2]:

orders = pd.read_excel("Company ShopX - Order Report.xlsx")
product_weight = pd.read_excel("Company ShopX - Product Weight.xlsx")
zone_map = pd.read_excel("Company ShopX - Warehouse&Customer Pin Code and Zone details.xlsx")
invoice = pd.read_excel("Courier Company - Invoice.xlsx")
rates = pd.read_excel("Courier Company - Rates.xlsx")

In [3]:
orders

,Order ID,Product Code,Units Ordered
0,2001827036,8904223818706,1.0
1,2001827036,8904223819093,1.0
2,2001827036,8904223819109,1.0
3,2001827036,8904223818430,1.0
4,2001827036,8904223819277,1.0
...,...,...,...
395,2001806229,8904223818942,1.0
396,2001806229,8904223818850,1.0
397,2001806226,8904223818850,2.0
398,2001806210,8904223816214,1.0


#### Calculate total order weight (ShopX side)

In [5]:
##MERGE orders WITH product_weight
orders = orders.merge(
    product_weight,
    on="Product Code",
    how="left"
)


#Calculate weight per row
orders["Total_Weight_g"] = orders["Units Ordered"] * orders["Product Weight (g)"]

#total weight per order
order_weight = orders.groupby("Order ID", as_index=False)["Total_Weight_g"].sum()

#Convert grams → KG
order_weight["Total Weight ShopX (KG)"] = order_weight["Total_Weight_g"] / 1000



### Convert weight to slab (0.5 KG rule)

In [6]:
order_weight["Weight Slab ShopX (KG)"] = np.ceil(
    order_weight["Total Weight ShopX (KG)"] * 2
) / 2


#### Merge ShopX weight with courier invoice

In [7]:
df = invoice.merge(order_weight, on="Order ID", how="left")


#### Courier weight & courier slab

In [8]:
df["Total Weight Courier (KG)"] = df["Chargeable Weight"] / 1000

df["Weight Slab Courier (KG)"] = np.ceil(
    df["Total Weight Courier (KG)"] * 2
) / 2


#### Find delivery zone as per ShopX

In [9]:
df = df.merge(
    zone_map,
    on=["Store House Pincode", "Customer Area Code"],
    how="left"
)


In [10]:
#Rename the zone columns
df = df.rename(columns={
    "Delivery Zone_x": "Delivery Zone Courier",
    "Delivery Zone_y": "Delivery Zone ShopX"
})


#### Prepare slab count for charge calculation

In [11]:
df["Slab_Count"] = df["Weight Slab ShopX (KG)"] / 0.5


In [14]:
#Prepare zone in lowercase
df["zone_lower"] = df["Delivery Zone ShopX"].str.lower()





#### Get FORWARD fixed & additional charges

In [15]:
#Forward FIXED
df["fwd_fixed"] = np.where(
    df["zone_lower"] == "a", rates.loc[0, "fwd_a_fixed"],
    np.where(df["zone_lower"] == "b", rates.loc[0, "fwd_b_fixed"],
    np.where(df["zone_lower"] == "c", rates.loc[0, "fwd_c_fixed"],
    np.where(df["zone_lower"] == "d", rates.loc[0, "fwd_d_fixed"],
             rates.loc[0, "fwd_e_fixed"])))
)

##Forward ADDITIONAL
df["fwd_additional"] = np.where(
    df["zone_lower"] == "a", rates.loc[0, "fwd_a_additional"],
    np.where(df["zone_lower"] == "b", rates.loc[0, "fwd_b_additional"],
    np.where(df["zone_lower"] == "c", rates.loc[0, "fwd_c_additional"],
    np.where(df["zone_lower"] == "d", rates.loc[0, "fwd_d_additional"],
             rates.loc[0, "fwd_e_additional"])))
)


#### Calculate FORWARD charge

In [16]:
df["Forward_Charge"] = np.where(
    df["Freight Type"].str.contains("Forward", case=False),
    df["fwd_fixed"] + (df["Slab_Count"] - 1) * df["fwd_additional"],
    0
)


#### Get RTO fixed & additional charges

In [17]:
#RTO FIXED
df["rto_fixed"] = np.where(
    df["zone_lower"] == "a", rates.loc[0, "rto_a_fixed"],
    np.where(df["zone_lower"] == "b", rates.loc[0, "rto_b_fixed"],
    np.where(df["zone_lower"] == "c", rates.loc[0, "rto_c_fixed"],
    np.where(df["zone_lower"] == "d", rates.loc[0, "rto_d_fixed"],
             rates.loc[0, "rto_e_fixed"])))
)

#RTO ADDITIONAL
df["rto_additional"] = np.where(
    df["zone_lower"] == "a", rates.loc[0, "rto_a_additional"],
    np.where(df["zone_lower"] == "b", rates.loc[0, "rto_b_additional"],
    np.where(df["zone_lower"] == "c", rates.loc[0, "rto_c_additional"],
    np.where(df["zone_lower"] == "d", rates.loc[0, "rto_d_additional"],
             rates.loc[0, "rto_e_additional"])))
)


#### Calculate RTO charge


In [18]:
df["RTO_Charge"] = np.where(
    df["Freight Type"].str.contains("RTO", case=False),
    df["rto_fixed"] + (df["Slab_Count"] - 1) * df["rto_additional"],
    0
)


#### Final expected charge

In [19]:
df["Expected Charge (Rs.)"] = df["Forward_Charge"] + df["RTO_Charge"]


####  Create the Difference column

In [20]:
df["Difference (Rs.)"] = df["Expected Charge (Rs.)"] - df["Total Amount (Rs.)"]
df["Difference (Rs.)"] = df["Difference (Rs.)"].round(2)


In [22]:
#Clean column names
df.columns = df.columns.str.strip()


#rename zone columns clearly
df.rename(columns={
    "AWB Code (Airway Bill Number)": "AWB Number"
}, inplace=True)

#Rename columns to match the exact PDF 
df = df.rename(columns={
    "Total Amount (Rs.)": "Charges Billed by Courier Company (Rs.)",
    "Difference (Rs.)": "Difference Between Expected Charges and Billed Charges (Rs.)"
})


In [23]:
print(df.columns)

Index(['AWB Number', 'Order ID', 'Chargeable Weight', 'Store House Pincode',
       'Customer Area Code', 'Delivery Zone Courier', 'Freight Type',
       'Charges Billed by Courier Company (Rs.)', 'Total_Weight_g',
       'Total Weight ShopX (KG)', 'Weight Slab ShopX (KG)',
       'Total Weight Courier (KG)', 'Weight Slab Courier (KG)',
       'Delivery Zone ShopX', 'Slab_Count', 'zone_lower', 'fwd_fixed',
       'fwd_additional', 'Forward_Charge', 'rto_fixed', 'rto_additional',
       'RTO_Charge', 'Expected Charge (Rs.)',
       'Difference Between Expected Charges and Billed Charges (Rs.)'],
      dtype='object')


### ONLY RENAME COLUMNS 

In [24]:
df = df.rename(columns={
    "Total Weight ShopX (KG)": "Total weight as per ShopX (KG)",
    "Weight Slab ShopX (KG)": "Weight slab as per ShopX (KG)",
    "Total Weight Courier (KG)": "Total weight as per Courier Company (KG)",
    "Weight Slab Courier (KG)": "Weight slab charged by Courier Company (KG)",
    "Delivery Zone ShopX": "Delivery Zone as per ShopX",
    "Delivery Zone Courier": "Delivery Zone charged by Courier Company",
    "Expected Charge (Rs.)": "Expected Charge as per ShopX (Rs.)"
})


#### Create Order-Level Output   --- Output Data 1

In [25]:
order_level_output = df[[
    "Order ID",
    "AWB Number",
    "Total weight as per ShopX (KG)",
    "Weight slab as per ShopX (KG)",
    "Total weight as per Courier Company (KG)",
    "Weight slab charged by Courier Company (KG)",
    "Delivery Zone as per ShopX",
    "Delivery Zone charged by Courier Company",
    "Expected Charge as per ShopX (Rs.)",
    "Charges Billed by Courier Company (Rs.)",
    "Difference Between Expected Charges and Billed Charges (Rs.)"
]]


#### OUTPUT DATA 2 — SUMMARY TABLE

In [26]:
#Create Charge Status
df["Charge Status"] = np.where(
    df["Difference Between Expected Charges and Billed Charges (Rs.)"] == 0,
    "Correctly Charged",
    np.where(
        df["Difference Between Expected Charges and Billed Charges (Rs.)"] < 0,
        "Overcharged",
        "Undercharged"
    )
)


##Create Summary Table
summary_output = df.groupby("Charge Status").agg(
    Count=("Order ID", "count"),
    Amount_Rs=("Charges Billed by Courier Company (Rs.)", "sum")
).reset_index()


In [27]:
# renames output
summary_output["Charge Status"] = summary_output["Charge Status"].replace({
    "Correctly Charged": "Total orders where ShopX has been correctly charged",
    "Overcharged": "Total Orders where ShopX has been overcharged",
    "Undercharged": "Total Orders where ShopX has been undercharged"
})

#### FINAL SUBMISSION — SAVE INTO EXCEL (TWO WORKBOOKS)

##### Save Order Level Output (Workbook 1)

In [29]:
order_level_output.to_excel(
    "ShopX_Order_Level_Calculation.xlsx",
    index=False
)


##### Save Summary Output (Workbook 2)

In [30]:
summary_output.to_excel(
    "ShopX_Summary_Table.xlsx",
    index=False
)


##### Load both output Excel files

In [31]:
order_df = pd.read_excel("ShopX_Order_Level_Calculation.xlsx")
summary_df = pd.read_excel("ShopX_Summary_Table.xlsx")


In [32]:
#Check structure of ORDER LEVEL output
print(order_df.columns)
print(order_df.head())


Index(['Order ID', 'AWB Number', 'Total weight as per ShopX (KG)',
       'Weight slab as per ShopX (KG)',
       'Total weight as per Courier Company (KG)',
       'Weight slab charged by Courier Company (KG)',
       'Delivery Zone as per ShopX',
       'Delivery Zone charged by Courier Company',
       'Expected Charge as per ShopX (Rs.)',
       'Charges Billed by Courier Company (Rs.)',
       'Difference Between Expected Charges and Billed Charges (Rs.)'],
      dtype='object')
     Order ID     AWB Number  Total weight as per ShopX (KG)  \
0  2001806232  1091117222124                           1.302   
1  2001806273  1091117222194                           0.615   
2  2001806408  1091117222931                           2.265   
3  2001806458  1091117223244                           0.700   
4  2001807012  1091117229345                           0.240   

   Weight slab as per ShopX (KG)  Total weight as per Courier Company (KG)  \
0                            1.5                